# Parse ENTOSE TYNDP 2022 data

Notebook to parse TYNDP 2022 data by ENTSOE:
https://2022.entsos-tyndp-scenarios.eu/

creates the following files in parsed_data/TYNDP_2022:
- load_tyndp22.csv with hourly load values for all countries
- cap_tyndp22.csv with yearly generation capacities for all countries
- ntc_tyndp22.csv with yearly ntc values (flow based and classic) for all countries

## Packages and options

In [8]:
import pandas as pd
import numpy as np
import datetime as dt

In [9]:
fn_in = "../../source_data/TYNDP_2022/Electricity_data_Visualisation_platform.xlsx"
fn_tech = "../../source_data/TYNDP_2020/tech_compare_tyndp.xlsx"
dir_out = "../../parsed_data/TYNDP_2022/"

Dictionary to rename country amd technology names from ENTSOE to own names

In [10]:
dict_country = {'AT00':'AT',
               'BE00':'BE',
               'BG00':'BG',
               'CH00':'CH',
               'CZ00':'CZ',
               'DE00':'DE',
               'DEKF':'DE',
               'DKE1':'DK',
               'DKKF':'DK',
               'DKW1':'DK',
               'ES00':'ES',
               'FI00':'FI',
               'FR00':'FR',
               'FR15':'FR',
               'GR00':'GR',
               'GR03':'GR',
               'HR00':'HR',
               'HU00':'HU',
               'IE00':'IE',
               'ITCN':'IT',
               'ITCO':'IT',
               'ITCS':'IT',
               'ITN1':'IT',
               'ITS1':'IT',
               'ITSA':'IT',
               'ITSI':'IT',
               'LUB1':'LU',
               'LUF1':'LU',
               'LUG1':'LU',
               'LUV1':'LU',
               'NL00':'NL',
               'NOM1':'NO',
               'NON1':'NO',
               'NOS0':'NO',
               'NOS1':'NO',
               'PL00':'PL',
               'PLE0':'PL',
               'PLI0':'PL',
               'PT00':'PT',
               'RO00':'RO',
               'SE01':'SE',
               'SE02':'SE',
               'SE03':'SE',
               'SE04':'SE',
               'SI00':'SI',
               'SK00':'SK',
               'UK00':'GB',
               'UKNI':'GB'
               }

In [11]:
dict_chp = {'Gas CHP':'Gas',
            'Other CHP':'Other'
           }

## Parse generation capacities

In [12]:
#technology matching
df_tech = pd.read_excel(fn_tech,sheet_name='tech_capacity_compare_tyndp').set_index(['tech capacity'])['model tech']
dict_tech_cap = df_tech.to_dict()

In [13]:
df_in = pd.read_excel(fn_in,sheet_name='Capacity & Dispatch')
df_in.head()


,Node/Line,Scenario,Year,Parameter,Climate Year,Fuel,Category,Value
0,AL00,Distributed Energy,2030,Capacity (MW),1995,Hydro,Electricity Market,2092.944092
1,AL00,Distributed Energy,2030,Capacity (MW),1995,Hydro,Electricity Market,611.812988
2,AL00,Distributed Energy,2030,Capacity (MW),1995,Gas,Electricity Market,200.000000
3,AL00,Distributed Energy,2030,Capacity (MW),1995,Gas,Electricity Market,100.000000
4,AL00,Distributed Energy,2030,Capacity (MW),1995,Solar,Electricity Market,330.000000


In [31]:
df_cap = df_in[(df_in.Parameter == 'Capacity (MW)') 
               & (df_in.Scenario == 'Global Ambition')
               & (df_in.Year == 2030)
               & (df_in['Climate Year'] == 1995)
               & (df_in.Category == 'Electricity Market')
               ]
df_cap.head()

,Node/Line,Scenario,Year,Parameter,Climate Year,Fuel,Category,Value
54,AL00,Global Ambition,2030,Capacity (MW),1995,Hydro,Electricity Market,2092.944092
55,AL00,Global Ambition,2030,Capacity (MW),1995,Hydro,Electricity Market,611.812988
56,AL00,Global Ambition,2030,Capacity (MW),1995,Gas,Electricity Market,200.000000
57,AL00,Global Ambition,2030,Capacity (MW),1995,Gas,Electricity Market,100.000000
58,AL00,Global Ambition,2030,Capacity (MW),1995,Solar,Electricity Market,330.000000


In [ ]:
df_cap["country"] = df_cap['Node/Line'].map(dict_country)
df_cap['technology'] = df_cap['Generator_ID'].map(dict_tech_cap)
df_cap['MW'] = df_cap['Value']
df_cap = df_cap[['country','technology','Scenario','Year','Climate Year','MW']]
df_cap = df_cap.rename(columns={'Scenario':'scenario','Year':'runyear','Climate Year':'climateyear'})
df_cap = df_cap.groupby(['scenario','runyear','climateyear','country','technology']).sum()
df_cap = df_cap.dropna()
df_cap.head()

## Parse volumes

In [7]:
#technology matching
df_tech = pd.read_excel(fn_tech,sheet_name='tech_gen_compare_tyndp').set_index(['tech generation'])['model tech']
dict_tech_gen = df_tech.to_dict()

In [8]:
df = pd.read_excel(fn_in,sheet_name='MarketRun')
df = df[df.Sector=='Electricity']
df["country"] = df['Node/Line'].map(dict_country)
df = df[['country','Generator_ID','Parameter','Scenario','Year','Climate Year','Value']]
df = df.rename(columns={'Scenario':'scenario','Year':'runyear','Climate Year':'climateyear'})
df.head()

,country,Generator_ID,Parameter,scenario,runyear,climateyear,Value
0,ES,DSR,Generation,Distributed Energy,2040,2007,460.0
1,FI,DSR,Generation,Distributed Energy,2040,2007,4925.0
2,AT,Gas CHP,Generation,Distributed Energy,2040,2007,4733.0
3,BE,Gas CHP,Generation,Distributed Energy,2040,2007,5433.0
4,CH,Gas CHP,Generation,Distributed Energy,2040,2007,3610.0


In [9]:
df_load1 = df[df.Parameter.isin(['Demand'])].copy()
df_load2 = df[df.Generator_ID.isin(['DSR'])].copy()
df_load = df_load1.append(df_load2)
df_load = df_load.pivot_table(index=['scenario','runyear','country','climateyear'],columns='Parameter',values='Value')
df_load = df_load.groupby(['scenario','runyear','country','climateyear']).sum()
df_load['Demand'] = df_load['Demand'] * 1000
df_load['Generation'] = df_load['Generation'] * 1000
df_load['Demand_DSR'] = df_load['Demand'] - df_load['Generation']
df_load = df_load.dropna()
df_load = df_load[['Demand','Demand_DSR']]
df_load.head()

Parameter                                             Demand    Demand_DSR
scenario           runyear country climateyear                            
Distributed Energy 2030    AT      1982         9.719566e+07  9.719566e+07
                                   1984         9.728581e+07  9.728581e+07
                                   2007         9.602100e+07  9.602100e+07
                           BE      1982         1.009248e+08  1.009248e+08
                                   1984         1.010277e+08  1.010277e+08

In [10]:
df_res = df[df.Parameter.isin(['Generation','Curtailed Energy'])].copy()
df_res['technology'] = df_res['Generator_ID'].map(dict_tech_gen)
df_res = df_res[df_res.technology.isin(['Solar', 'RunOfRiver', 'Biomass', 'WindOffshore', 'WindOnshore','Other'])]
df_res = df_res.rename(columns={'Scenario':'scenario','Year':'runyear','Climate Year':'climateyear'})
df_res = df_res.groupby(['country','technology','scenario','runyear','climateyear']).sum()
df_res['MWh'] = df_res['Value'] * 1000
df_res = df_res[['MWh']]
df_res.head()

MWh
country technology scenario           runyear climateyear              
AT      Biomass    Distributed Energy 2030    1982         3.372631e+06
                                              1984         3.372631e+06
                                              2007         3.372000e+06
                                      2040    1982         3.372631e+06
                                              1984         3.372631e+06

In [35]:
df_gen = df[df.Parameter.isin(['Generation','Curtailed Energy'])].copy()
df_gen['technology'] = df_gen['Generator_ID'].map(dict_tech_gen)
df_gen = df_gen.rename(columns={'Scenario':'scenario','Year':'runyear','Climate Year':'climateyear'})
df_gen = df_gen.groupby(['country','technology','scenario','runyear','climateyear']).sum()
df_gen['MWh'] = df_gen['Value'] * 1000
df_gen = df_gen[['MWh']]
df_gen.head()

MWh
country technology scenario           runyear climateyear              
AT      Battery    Distributed Energy 2030    1982         3.050376e+05
                                              1984         3.029840e+05
                                              2007         3.340000e+05
                                      2040    1982         2.123303e+06
                                              1984         2.110534e+06

In [36]:
df_emissions = df[df.Parameter.isin(['Emissions'])].copy()
df_emissions['technology'] = df_emissions['Generator_ID'].map(dict_tech_gen)
df_emissions = df_emissions.rename(columns={'Scenario':'scenario','Year':'runyear','Climate Year':'climateyear','Value':'emissions'})
df_emissions = df_emissions.groupby(['country','technology','scenario','runyear','climateyear']).sum()
df_emissions.head()

emissions
country technology scenario           runyear climateyear              
AT      Gas        Distributed Energy 2030    1982         3.911628e+06
                                              1984         3.721443e+06
                                              2007         3.362309e+06
                                      2040    1982         8.851256e+05
                                              1984         8.238326e+05

In [37]:
df_gen = df_gen.merge(df_emissions, right_index=True, left_index=True)
df_gen['emission factor'] = df_gen['emissions']/df_gen['MWh']
df_gen.head()

MWh  \
country technology scenario           runyear climateyear                 
AT      Gas        Distributed Energy 2030    1982         9.079201e+06   
                                              1984         8.548025e+06   
                                              2007         7.535270e+06   
                                      2040    1982         5.893345e+06   
                                              1984         5.717558e+06   

                                                              emissions  \
country technology scenario           runyear climateyear                 
AT      Gas        Distributed Energy 2030    1982         3.911628e+06   
                                              1984         3.721443e+06   
                                              2007         3.362309e+06   
                                      2040    1982         8.851256e+05   
                                              1984         8.238326e+05   

                                                           emission factor  
country technology scenario           runyear climateyear                   
AT      Gas        Distributed Energy 2030    1982                0.430834  
                                              1984                0.435357  
                                              2007                0.446209  
                                      2040    1982                0.150191  
                                              1984                0.144088

In [12]:
df_chp = df[df.Generator_ID.isin(["Gas CHP","Other CHP"]) & df.Parameter.isin(['Generation'])].copy()
df_chp['technology'] = df_chp['Generator_ID'].map(dict_chp)
df_chp = df_chp.rename(columns={'Scenario':'scenario','Year':'runyear','Climate Year':'climateyear'})
df_chp = df_chp.groupby(['country','technology','scenario','runyear','climateyear']).sum()
df_chp['Value'] = df_chp['Value'] * 1000
df_chp.head()

Value
country technology scenario           runyear climateyear              
AT      Gas        Distributed Energy 2030    1982         4.733976e+06
                                              1984         4.733976e+06
                                              2007         4.733000e+06
                                      2040    1982         4.733976e+06
                                              1984         4.733976e+06

## Parse NTC values

In [13]:
df_ntc = pd.read_excel(fn_in,sheet_name='Line')
df_ntc[['fromnode','tonode']] = df_ntc['Node/Line'].str.split(pat='-',expand=True) 
df_ntc['from'] = df_ntc['fromnode'].map(dict_country) 
df_ntc['to'] = df_ntc['tonode'].map(dict_country) 
df_ntc = df_ntc[df_ntc.Case == 'Reference Grid']
df_ntc = df_ntc[df_ntc['from'] != df_ntc['to']]
df_ntc = df_ntc.rename(columns={'Scenario':'scenario','Year':'runyear','Climate Year':'climateyear'})
df_ntc = df_ntc.groupby(['from','to','scenario','runyear','climateyear','Parameter']).sum()
df_ntc = df_ntc.reset_index().pivot_table(index = ['from','to','scenario','runyear','climateyear'],
                           columns = 'Parameter',
                           values = 'Value')
df_ntc['Flow Back'] = df_ntc['Flow Back'].fillna(df_ntc['Flow'])
df_ntc['Import Capacity'] = df_ntc['Import Capacity'].fillna(- df_ntc['Export Capacity'])
df_ntc['Import Capacity'] = -df_ntc['Import Capacity']
df_ntc = df_ntc.dropna()
df_ntc.head()

Parameter                                       Export Capacity         Flow  \
from to scenario           runyear climateyear                                 
AT   CH Distributed Energy 2030    1982                  1200.0   577.879265   
                                   1984                  1200.0   604.262596   
                                   2007                  1200.0   819.000000   
                           2040    1982                  1200.0  2075.039579   
                                   1984                  1200.0  2196.132057   

Parameter                                         Flow Back  Import Capacity  
from to scenario           runyear climateyear                                
AT   CH Distributed Energy 2030    1982         4193.494736           1200.0  
                                   1984         4145.006943           1200.0  
                                   2007         4339.000000           1200.0  
                           2040    1982         2612.392183           1200.0  
                                   1984         2404.323422           1200.0

## Parse load time series

In [14]:
df_load_NT2025 = pd.read_csv(fn_load_NT2025,sep=",")
df_load_NT2025['scenario'] = 'National Trends'
df_load_NT2025['year'] = 2025
df_load_NT2025['month'] = df_load_NT2025['PATTERN'].str[1:3].astype(int)
df_load_NT2025['day'] = df_load_NT2025['PATTERN'].str[5:7].astype(int)
df_load_NT2025['hour'] = df_load_NT2025['PATTERN'].str[9:].astype(int) - 1
df_load_NT2025['time'] = df_load_NT2025[['year', 'month', 'day', 'hour']].apply(lambda s : dt.datetime(*s),axis = 1)
df_load_NT2025 = df_load_NT2025.drop(columns=['month','day','hour','PATTERN'])
df_load_NT2025 = df_load_NT2025.melt(id_vars=['scenario','year', 'time'])
df_load_NT2025['country'] = df_load_NT2025['variable'].map(dict_country)
df_load_NT2025 = df_load_NT2025.groupby(['scenario','year','country','time']).sum()
df_load_NT2025.head(1)

,,,,value
scenario,year,country,time,
National Trends,2025,AT,2025-01-01,7320.769525


In [15]:
df_load_NT2030 = pd.read_csv(fn_load_NT2030,sep=",")
df_load_NT2030['scenario'] = 'National Trends'
df_load_NT2030['year'] = 2030
df_load_NT2030['month'] = df_load_NT2030['PATTERN'].str[1:3].astype(int)
df_load_NT2030['day'] = df_load_NT2030['PATTERN'].str[5:7].astype(int)
df_load_NT2030['hour'] = df_load_NT2030['PATTERN'].str[9:].astype(int) - 1
df_load_NT2030['time'] = df_load_NT2030[['year', 'month', 'day', 'hour']].apply(lambda s : dt.datetime(*s),axis = 1)
df_load_NT2030 = df_load_NT2030.drop(columns=['month','day','hour','PATTERN'])
df_load_NT2030 = df_load_NT2030.melt(id_vars=['scenario','year', 'time'])
df_load_NT2030['country'] = df_load_NT2030['variable'].map(dict_country)
df_load_NT2030 = df_load_NT2030.groupby(['scenario','year','country','time']).sum()
df_load_NT2030.head(1)

,,,,value
scenario,year,country,time,
National Trends,2030,AT,2030-01-01,7671.244149


In [16]:
df_load_NT2040 = pd.read_csv(fn_load_NT2040,sep=",")
df_load_NT2040['scenario'] = 'National Trends'
df_load_NT2040['year'] = df_load_NT2040['YEAR'] - 1
df_load_NT2040['Period'] = df_load_NT2040['Period'] -1
df_load_NT2040['time'] = df_load_NT2040[['year','MONTH','DAY','Period']].apply(lambda s : dt.datetime(*s),axis = 1)
df_load_NT2040 = df_load_NT2040.drop(columns=['MONTH','DAY','Period','YEAR'])
df_load_NT2040 = df_load_NT2040.melt(id_vars=['scenario','year', 'time'])
df_load_NT2040['country'] = df_load_NT2040['variable'].map(dict_country)
df_load_NT2040 = df_load_NT2040.groupby(['scenario','year','country','time']).sum()
df_load_NT2040.head(1)

,,,,value
scenario,year,country,time,
National Trends,2040,AT,2040-01-01,8900.134943


In [17]:
df_load_DE2030 = pd.read_csv(fn_load_DE2030,sep=",")
df_load_DE2030 = df_load_DE2030.dropna()
df_load_DE2030['scenario'] = 'Distributed Energy'
df_load_DE2030['year'] = 2030
df_load_DE2030['month'] = df_load_DE2030['MONTH'].astype(int)
df_load_DE2030['day'] = df_load_DE2030['DAY'].astype(int)
df_load_DE2030['hour'] = df_load_DE2030['Period'].astype(int) -1
df_load_DE2030['time'] = df_load_DE2030[['year','month','day','hour']].apply(lambda s : dt.datetime(*s),axis = 1)
df_load_DE2030 = df_load_DE2030.drop(columns=['YEAR','MONTH','DAY','Period','month','day','hour'])
df_load_DE2030 = df_load_DE2030.melt(id_vars=['scenario','year', 'time'])
df_load_DE2030['country'] = df_load_DE2030['variable'].map(dict_country)
df_load_DE2030 = df_load_DE2030.groupby(['scenario','year','country','time']).sum()
df_load_DE2030.head(1)

,,,,value
scenario,year,country,time,
Distributed Energy,2030,AT,2030-01-01,9314.949567


In [18]:
df_load_DE2040 = pd.read_csv(fn_load_DE2040,sep=",")
df_load_DE2040 = df_load_DE2040.drop(columns=['Unnamed: 60'])
df_load_DE2040 = df_load_DE2040.dropna()
df_load_DE2040['scenario'] = 'Distributed Energy'
df_load_DE2040['year'] = 2040
df_load_DE2040['month'] = df_load_DE2040['MONTH'].astype(int)
df_load_DE2040['day'] = df_load_DE2040['DAY'].astype(int)
df_load_DE2040['hour'] = df_load_DE2040['Period'].astype(int) -1
df_load_DE2040['time'] = df_load_DE2040[['year','month','day','hour']].apply(lambda s : dt.datetime(*s),axis = 1)
df_load_DE2040 = df_load_DE2040.drop(columns=['YEAR','MONTH','DAY','Period','month','day','hour'])
df_load_DE2040 = df_load_DE2040.melt(id_vars=['scenario','year', 'time'])
df_load_DE2040['country'] = df_load_DE2040['variable'].map(dict_country)
df_load_DE2040 = df_load_DE2040.groupby(['scenario','year','country','time']).sum()
df_load_DE2040.head(1)

,,,,value
scenario,year,country,time,
Distributed Energy,2040,AT,2040-01-01,13626.51516


In [19]:
df_load_GA2030 = pd.read_csv(fn_load_GA2030,sep=",")
df_load_GA2030 = df_load_GA2030.dropna()
df_load_GA2030['scenario'] = 'Global Ambition'
df_load_GA2030['year'] = 2030
df_load_GA2030['month'] = df_load_GA2030['MONTH'].astype(int)
df_load_GA2030['day'] = df_load_GA2030['DAY'].astype(int)
df_load_GA2030['hour'] = df_load_GA2030['Period'].astype(int) -1
df_load_GA2030['time'] = df_load_GA2030[['year','month','day','hour']].apply(lambda s : dt.datetime(*s),axis = 1)
df_load_GA2030 = df_load_GA2030.drop(columns=['YEAR','MONTH','DAY','Period','month','day','hour'])
df_load_GA2030 = df_load_GA2030.melt(id_vars=['scenario','year', 'time'])
df_load_GA2030['country'] = df_load_GA2030['variable'].map(dict_country)
df_load_GA2030 = df_load_GA2030.groupby(['scenario','year','country','time']).sum()
df_load_GA2030.head(1)

,,,,value
scenario,year,country,time,
Global Ambition,2030,AT,2030-01-01,9138.781478


In [20]:
df_load_GA2040 = pd.read_csv(fn_load_GA2040,sep=",")
df_load_GA2040 = df_load_GA2040.dropna()
df_load_GA2040['scenario'] = 'Global Ambition'
df_load_GA2040['year'] = 2040
df_load_GA2040['month'] = df_load_GA2040['MONTH'].astype(int)
df_load_GA2040['day'] = df_load_GA2040['DAY'].astype(int)
df_load_GA2040['hour'] = df_load_GA2040['Period'].astype(int) -1
df_load_GA2040['time'] = df_load_GA2040[['year','month','day','hour']].apply(lambda s : dt.datetime(*s),axis = 1)
df_load_GA2040 = df_load_GA2040.drop(columns=['YEAR','MONTH','DAY','Period','month','day','hour'])
df_load_GA2040 = df_load_GA2040.melt(id_vars=['scenario','year', 'time'])
df_load_GA2040['country'] = df_load_GA2040['variable'].map(dict_country)
df_load_GA2040 = df_load_GA2040.groupby(['scenario','year','country','time']).sum()
df_load_GA2040.head(1)

,,,,value
scenario,year,country,time,
Global Ambition,2040,AT,2040-01-01,12431.6853


In [21]:
df_load_ts = pd.DataFrame()
df_load_ts = df_load_ts.append(df_load_NT2025)
df_load_ts = df_load_ts.append(df_load_NT2030)
df_load_ts = df_load_ts.append(df_load_NT2040)
df_load_ts = df_load_ts.append(df_load_DE2030)
df_load_ts = df_load_ts.append(df_load_DE2040)
df_load_ts = df_load_ts.append(df_load_GA2030)
df_load_ts = df_load_ts.append(df_load_GA2040)
df_load_ts.head()

value
scenario        year country time                            
National Trends 2025 AT      2025-01-01 00:00:00  7320.769525
                             2025-01-01 01:00:00  7000.279843
                             2025-01-01 02:00:00  6739.479427
                             2025-01-01 03:00:00  6517.561699
                             2025-01-01 04:00:00  6221.576044

compare to yearly values

In [22]:
df_load_ts_year = df_load_ts.reset_index().groupby(['scenario','year','country']).sum().reset_index()
df_load_compare = df_load_ts_year.merge(df_load.reset_index(),left_on = ['scenario','year','country'],right_on = ['scenario','runyear','country'])
df_load_compare['from hourly'] = df_load_compare.value
df_load_compare = df_load_compare[df_load_compare['climateyear']==1984].set_index(['scenario','year','country'])
df_load_compare = df_load_compare[['from hourly','Demand','Demand_DSR']]
df_load_compare.head(40)

from hourly        Demand    Demand_DSR
scenario           year country                                          
Distributed Energy 2030 AT       8.889412e+07  9.728581e+07  9.728581e+07
                        BE       1.010876e+08  1.010277e+08  1.010277e+08
                        BG       3.150482e+07  3.139906e+07  3.139906e+07
                        CH       6.097359e+07  6.079236e+07  6.079236e+07
                        CZ       7.334794e+07  7.309859e+07  7.309859e+07
                        DE       6.880964e+08  7.038720e+08  7.038720e+08
                        DK       4.863682e+07  2.424287e+07  2.424287e+07
                        ES       2.710124e+08  2.841102e+08  2.841089e+08
                        FI       1.107885e+08  1.104323e+08  1.094350e+08
                        FR       4.858646e+08  2.420703e+08  2.420703e+08
                        GB       3.702926e+08  1.846233e+08  1.846233e+08
                        GR       6.294453e+07  3.138589e+07  3.138589e+07
                        HR       1.897793e+07  1.891761e+07  1.891761e+07
                        HU       4.842033e+07  4.493945e+07  4.493945e+07
                        IE       4.256503e+07  4.244450e+07  4.244450e+07
                        IT       3.403458e+08  5.658375e+07  5.658375e+07
                        LU       9.139050e+06  3.466151e+06  3.466151e+06
                        NL       1.376483e+08  1.372761e+08  1.372761e+08
                        NO       1.509437e+08  5.013590e+07  5.013175e+07
                        PL       1.785993e+08  1.781186e+08  1.781186e+08
                        PT       5.351256e+07  5.335625e+07  5.335625e+07
                        RO       6.517555e+07  6.495375e+07  6.495375e+07
                        SE       1.598113e+08  3.982117e+07  3.982117e+07
                        SI       1.494167e+07  1.489565e+07  1.489565e+07
                        SK       3.032731e+07  3.024953e+07  3.024953e+07
                   2040 AT       1.111686e+08  1.194335e+08  1.194335e+08
                        BE       1.176325e+08  1.175393e+08  1.175343e+08
                        BG       3.387172e+07  3.376302e+07  3.376302e+07
                        CH       5.664417e+07  5.647229e+07  5.647229e+07
                        CZ       8.476356e+07  8.446438e+07  8.446438e+07
                        DE       7.889331e+08  8.131406e+08  8.131406e+08
                        DK       6.344774e+07  3.162107e+07  3.162107e+07
                        ES       3.407400e+08  3.396538e+08  3.390670e+08
                        FI       1.499408e+08  1.494753e+08  1.427598e+08
                        FR       5.639371e+08  2.810277e+08  2.810277e+08
                        GB       3.945641e+08  2.108220e+08  2.108220e+08
                        GR       7.139479e+07  3.557215e+07  3.557215e+07
                        HR       2.187931e+07  2.181043e+07  2.181043e+07
                        HU       5.808341e+07  5.308833e+07  5.308833e+07
                        IE       4.859974e+07  4.846348e+07  4.846313e+07

## Export data

In [38]:
df_load_ts.to_csv(dir_out + "load_tyndp20.csv", encoding="utf-8", index=True)
df_res.to_csv(dir_out + "res_tyndp20.csv", encoding="utf-8", index=True)
df_cap.to_csv(dir_out + "cap_tyndp20.csv", encoding="utf-8", index=True)
df_ntc.to_csv(dir_out + "ntc_tyndp20.csv", encoding="utf-8", index=True)
df_chp.to_csv(dir_out + "chp_tyndp20.csv", encoding="utf-8", index=True)
df_gen.to_csv(dir_out + "gen_tyndp20.csv", encoding="utf-8", index=True)

In [24]:
df_load_compare.to_csv("../tyndp_2020_load_comparison.csv", encoding="utf-8", index=True)